# P1C — Colab Foundation Verification Gate

This notebook verifies the frozen dataset, cross-validation split, and deterministic seed registry in Google Colab. It produces only non-result-bearing provenance evidence and does not execute scientific models or imbalance-treatment methods.

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')
print('DRIVE_MOUNT=PASS')


## Configuration
Edit only `DATASET_PATH` and, if desired, `OUTPUT_DIR`. The uploaded CSV must be the exact dataset frozen by P1. Keep the GitHub token in Colab Secrets as `GITHUB_TOKEN`; do not paste it into a notebook cell.

In [ ]:
from pathlib import Path

REPO_URL = 'https://github.com/ArashSalehpourac/BreastCancer-Imbalance-Benchmark_35.git'
REPO_REF = 'main'
FOUNDATION_BASE_COMMIT = '2e6c405c6dce747fc86e3d72252f7a78831f1771'
REPO_DIR = Path('/content/BreastCancer-Imbalance-Benchmark_35')

# Change this line if you place the CSV elsewhere in My Drive.
DATASET_PATH = Path('/content/drive/MyDrive/BreastCancer-Imbalance-Benchmark_35/01_Experiment_Evidence/00_Dataset/wdbc.csv')
OUTPUT_DIR = Path('/content/drive/MyDrive/BreastCancer-Imbalance-Benchmark_35/01_Experiment_Evidence/P1C_Colab_Foundation')

print(f'DATASET_PATH={DATASET_PATH}')
print(f'OUTPUT_DIR={OUTPUT_DIR}')


In [ ]:
import os
import shutil
import subprocess

token = userdata.get('GITHUB_TOKEN')
if not token:
    raise RuntimeError('Add a read-only GitHub token to Colab Secrets with the name GITHUB_TOKEN.')

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

git_env = os.environ.copy()
git_env['GIT_CONFIG_COUNT'] = '1'
git_env['GIT_CONFIG_KEY_0'] = 'http.extraHeader'
git_env['GIT_CONFIG_VALUE_0'] = f'AUTHORIZATION: bearer {token}'
subprocess.run(['git', 'clone', '--branch', REPO_REF, REPO_URL, str(REPO_DIR)], check=True, env=git_env)

repo_head = subprocess.check_output(['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'], text=True).strip()
ancestor = subprocess.run(
    ['git', '-C', str(REPO_DIR), 'merge-base', '--is-ancestor', FOUNDATION_BASE_COMMIT, repo_head],
    check=False,
).returncode
if ancestor != 0:
    raise RuntimeError(f'Checked-out repository head {repo_head} does not descend from frozen foundation base {FOUNDATION_BASE_COMMIT}.')

print(f'REPOSITORY_HEAD={repo_head}')
print(f'FOUNDATION_BASE_COMMIT={FOUNDATION_BASE_COMMIT}')
print('REPOSITORY_ANCESTRY_GATE=PASS')
del token


In [ ]:
import sys
import subprocess

subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', str(REPO_DIR)], check=True)
print('FOUNDATION_PACKAGE_INSTALL=PASS')


In [ ]:
if not DATASET_PATH.is_file():
    raise FileNotFoundError(
        f'Dataset not found at {DATASET_PATH}. Upload the exact WDBC CSV to Drive and edit DATASET_PATH.'
    )
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('DRIVE_PATH_GATE=PASS')


In [ ]:
import subprocess
import sys

command = [
    sys.executable,
    str(REPO_DIR / 'scripts' / 'verify_colab_foundation.py'),
    '--dataset', str(DATASET_PATH),
    '--lock', str(REPO_DIR / 'data' / 'registry' / 'FOUNDATION_LOCK_v1.json'),
    '--output-dir', str(OUTPUT_DIR),
    '--git-commit', repo_head,
]
subprocess.run(command, cwd=REPO_DIR, check=True)


In [ ]:
evidence_files = sorted(path.name for path in OUTPUT_DIR.iterdir() if path.is_file())
print('EVIDENCE_FILES=')
for name in evidence_files:
    print(f'  {name}')
required = {
    'colab_foundation_gate.json',
    'colab_foundation_gate.json.sha256',
    'colab_foundation_manifest.json',
    'colab_foundation_manifest.json.sha256',
}
missing = required - set(evidence_files)
if missing:
    raise RuntimeError(f'Missing expected foundation evidence files: {sorted(missing)}')
print('COLAB_EVIDENCE_WRITE_GATE=PASS')
print('RESULT_BEARING=false')
print('COLAB_FOUNDATION_GATE=PASS')


## Stop here
A PASS only confirms the execution foundation. It does **not** authorize or run any scientific experiment. Do not add training, resampling, synthetic generation, evaluation, or manuscript-result cells to this notebook.